In [3]:
spike_files = [r"E:\Git\RFArduinoData\example_scripts\exp_data\h5_correlated\2025-11-26T16-29-46_RecID-1108_45343_Typ5-18h-18I_pois008_corr075_line_5p8V_DIV27_HS259_2_spikesonly.h5"]

# weird factor Cyprian figured out
T = 2*(2**31 / 50000) / 2

In [4]:
import h5py
import numpy as np

def load_h5_to_dict(filename):
    with h5py.File(filename, "r") as f:
        data = {}
        for key in f.keys():
            value = f[key][()]
            if isinstance(value, np.ndarray):
                data[key] = value
            else:
                data[key] = value[0]
        return data
    
def unwrap(x, T, jump_thresh=None):
    x = np.asarray(x, dtype=np.float64)
    if x.size < 2:
        return x

    if jump_thresh is None:
        jump_thresh = -0.5 * T

    dx = np.diff(x)
    wrap = dx < jump_thresh
    nwrap = np.cumsum(np.r_[0, wrap.astype(np.int64)])
    return x + (2.0 * nwrap * T)

for spike_file in spike_files:
    fixed_file = spike_file.replace(".h5", "_unwrap.h5")

    with h5py.File(spike_file, "r") as fin, h5py.File(fixed_file, "w") as fout:
        for k, v in fin.attrs.items():
            fout.attrs[k] = v

        for name, obj in fin.items():
            if isinstance(obj, h5py.Dataset):
                data = obj[()]
                if name.startswith("Channel_") and isinstance(data, np.ndarray) and data.dtype == np.float64 and data.ndim == 1:
                    data = unwrap(data, T)

                ds = fout.create_dataset(name, data=data)
                for k, v in obj.attrs.items():
                    ds.attrs[k] = v
            else:
                fin.copy(obj, fout, name)

    print("Wrote:", fixed_file)

Wrote: E:\Git\RFArduinoData\example_scripts\exp_data\h5_correlated\2025-11-26T16-29-46_RecID-1108_45343_Typ5-18h-18I_pois008_corr075_line_5p8V_DIV27_HS259_2_spikesonly_unwrap.h5
